# Task 2.1 — Dataset Selection and Setup

## Dataset: `make_moons` (scikit-learn)

**What the dataset is:** `sklearn.datasets.make_moons` generates 400 samples of two interleaving crescent-shaped (non-linearly separable) binary classes in 2D, with Gaussian noise (σ = 0.25) added to each point. The dataset is synthetically generated, requires no download, and has no licensing restrictions.

**Why it is a reasonable testbed for iSVM:** The paper's core claim is that a DP mixture of local SVMs handles data with multiple underlying clusters better than a single global SVM. The two-moons data naturally embodies this: the top crescent and the bottom crescent form two spatially separated clusters, each of which can be (approximately) separated by a local linear classifier. iSVM is expected to discover the two clusters via the DP feature mixture and fit one SVM per cluster—exactly mirroring the binary experiment in Figure 1 of the paper, where data is sampled from a mixture of two Gaussians.

**Limitations vs. the paper's dataset:** The original paper uses a 4-covariate Gaussian mixture (Setting 1) with 10 000 samples and a non-linear Bernoulli response function (ϑ = −a sin(x₁³ + 1.2) − …), and a 634-dimensional Flickr image dataset. Our 2D toy dataset is far lower-dimensional and smaller (400 samples), which means the DP mixture will converge faster but the advantage of nonlinear local experts over a single global RBF-SVM will be less pronounced. We also do not reproduce the multi-class (L > 2) scenario.

**Preprocessing steps:** StandardScaler is applied to zero-mean and unit-variance scale both features, matching the paper's use of normalised inputs and ensuring fair comparison of kernel bandwidths across methods.


In [1]:
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Random seed for reproducibility (set at top of notebook)
np.random.seed(42)

# --- Generate dataset ---
X_raw, y = make_moons(n_samples=400, noise=0.25, random_state=42)

# --- Preprocessing: StandardScaler ---
scaler = StandardScaler()
X = scaler.fit_transform(X_raw)

# --- Train/test split (70/30, stratified) ---
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y)

print(f"Total samples : {len(X)}")
print(f"Training set  : {X_train.shape}  classes: {np.bincount(y_train)}")
print(f"Test set      : {X_test.shape}   classes: {np.bincount(y_test)}")
print(f"Features      : {X_train.shape[1]} (2D, continuous)")


Total samples : 400
Training set  : (280, 2)  classes: [140 140]
Test set      : (120, 2)   classes: [60 60]
Features      : 2 (2D, continuous)


### Visualisation of Dataset

In [1]:
fig, ax = plt.subplots(figsize=(7, 5))
colors = ['#e74c3c', '#3498db']
markers = ['s', '*']
for cls, col, mk in zip([0, 1], colors, markers):
    mask = y_train == cls
    ax.scatter(X_train[mask, 0], X_train[mask, 1],
               c=col, marker=mk, s=50, label=f'Class {cls}',
               edgecolors='k', linewidths=0.4)
ax.set_title('Training Data: make_moons (n=280, noise=0.25)', fontsize=12)
ax.set_xlabel('Feature 1 (scaled)'); ax.set_ylabel('Feature 2 (scaled)')
ax.legend()
plt.tight_layout()
plt.savefig('/home/claude/partB/results/dataset_plot.png', dpi=150)
print("Dataset saved.")


Dataset saved.
